In [138]:
import pandas as pd
# import fireducks.pandas as pd
import numpy as np
LOWER_T_VALUE = 6*2
LAT_CELLS = 300
LON_CELLS = 300

LAT_RANGE = 300 - 0
LON_RANGE = 300 - 0

parquet = '../output/models/custom_ds_latent_size_128a_fivo_adamw/logprob.parquet'
def get_bin_map():
    _ds = pd.read_parquet(parquet, columns=['log_prob'])
    lat_lon_ds = pd.read_parquet('../new_data/ais_test_by_bins.parquet', columns=['latitude', 'longitude'])
    _ds = pd.merge(_ds, lat_lon_ds, how='inner', left_index=True, right_index=True)
    _ds = _ds.reset_index()
    _ds = _ds.set_index('latitude')
    _ds = _ds.set_index('longitude', append=True)
    # higher than a timestamp
    _ds = _ds[_ds['t']>= LOWER_T_VALUE]
    _ds = _ds.drop(['t', 'track_id'], axis=1)
    quantile=1.64
    _ds = _ds[_ds.groupby(level=[0,1], sort=True)['log_prob'].transform(lambda ds: ( ds - ds.mean()) <= quantile * ds.std())]
    _ds = _ds.sort_index()
    return _ds

def get_test_set():
    _ds = pd.read_parquet(parquet, columns=['log_prob'])
    lat_lon_ds = pd.read_parquet('../new_data/ais_test_by_bins.parquet', columns=['latitude', 'longitude'])
    _ds = pd.merge(_ds, lat_lon_ds, how='inner', left_index=True, right_index=True)
    return _ds

In [139]:
input = get_test_set()
input = input.reset_index()
input = input.set_index('latitude')
input = input.set_index('longitude', append=True)
binmap = get_bin_map()
binmap = binmap.rename({'log_prob':'sample'}, axis=1)

In [140]:
from scipy import stats
import numpy as np


def eval_cdf(sample, value):
    return stats.gaussian_kde(sample).integrate_box_1d(-np.inf, value)

# TODO: something in the join fails i think
binmap_dist = binmap.groupby(level=[0,1]).agg(dist_sample=('sample',lambda sample: sample.to_list()))
input = pd.merge(input, binmap_dist, how='inner', left_index=True, right_index=True)
input['anomaly'] = (input.apply(lambda row: eval_cdf(row['dist_sample'], row['log_prob']), axis=1) < 0.1)

In [141]:
input = input.reset_index()
input = input.set_index('track_id')
input = input.set_index('t', append=True)
input

latitude  longitude   log_prob  \
track_id t                                    
3147     0        213        110 -49.487236   
         1        212        109 -58.450813   
         2        210        108 -46.560780   
         3        208        107 -51.277756   
         4        207        107 -42.681595   
...               ...        ...        ...   
11536    27       213         30 -36.707634   
         28       211         32 -14.695847   
         29       210         34 -17.283325   
         30       209         35 -10.318414   
         31       209         35 -16.975281   

                                                   dist_sample  anomaly  
track_id t                                                               
3147     0   [-69.62775421142578, -22.12520980834961, -28.7...    False  
         1   [-48.74060821533203, -18.204715728759766, -16....     True  
         2   [-19.004623413085938, -14.742228507995605, -47...    False  
         3   [-40.736507415771484, -19.536388397216797, -36...     True  
         4   [-27.6734676361084, -19.657888412475586, -49.6...    False  
...                                                        ...      ...  
11536    27  [-18.382776260375977, -7.858282089233398, -36....    False  
         28  [-22.245304107666016, -48.34974670410156, -17....    False  
         29  [-42.2671012878418, -47.91305160522461, -45.35...    False  
         30  [-47.7331428527832, -10.318413734436035, -16.9...    False  
         31  [-47.7331428527832, -10.318413734436035, -16.9...    False  

[91923 rows x 5 columns]

In [137]:
from functools import reduce
def nCr_old(n, r):
    """Function calculates the number of combinations (n choose r)"""
    r = min(r, n-r)
    numer = reduce(op.mul, range(n, n-r, -1), 1)
    denom = reduce(op.mul, range(1, r+1), 1)
    return numer//denom


def NFA_old(ns,k):
    """Number of False Alarms"""
    B = 0
    for t in range(k,ns+1):
        B += nCr_old(ns,t)*(0.1**t)*(0.9**(ns-t))
        print(B)
    return 300*B


def contrario_detection_old(v_A_,epsilon=0.0091):
    """
    A contrario detection algorithms
    INPUT:
        v_A_: abnormal point indicator vector
        epsilon: threshold
    OUTPUT:
        v_anomalies: abnormal segment indicator vector

    """
    v_anomalies = np.zeros(len(v_A_))
    max_seq_len = min(MAX_SEQUENCE_LENGTH, len(v_A_))
    for d_ns in range(max_seq_len,0,-1):
        for d_ci in range(max_seq_len+1-d_ns):
            v_xi = v_A_[d_ci:d_ci+d_ns]
            d_k_xi = int(np.count_nonzero(v_xi))
            if NFA_old(d_ns,d_k_xi)<epsilon:
                v_anomalies[d_ci:d_ci+d_ns] = 1
    return v_anomalies



In [131]:
nCr = np.diag(np.array(list(1 for _ in range(MAX_SEQUENCE_LENGTH+1)), dtype=int))
nCr[:,0] = 1
for i in range(1, len(nCr)):
    for j in range(1, i):
        nCr[i,j] = nCr[i-1, j-1]+nCr[i-1, j]

for i in range(nCr.shape[0]):
    for j in range(i+1):
        assert nCr[i,j] == nCr_old(i,j), f'{i}, {j}'


# compute exponents
nCr_mul = np.zeros_like(nCr, dtype=np.float32)
for ns in range(len(nCr)):
    for t in range(ns+1):
        nCr_mul[ns,t] = nCr[ns,t]*(0.1**t)*(0.9**(ns-t))

# build NFA
NFA = np.zeros_like(nCr, dtype=np.float32)
for ns in range(NFA.shape[0]):
    for k in range(ns):
        NFA[ns, k] = 300*np.sum(nCr_mul[ns, k:(ns+1)])

In [132]:
NFA[0,0]



np.float32(0.0)

In [135]:
for i in range(1,NFA.shape[0]):
    for j in range(1,i):
        assert NFA[i,j] == NFA_old(i,j), f'{i}, {j}'

0.18000000000000002
0.19000000000000003


AssertionError: 2, 1

In [65]:

import numpy as np
import operator as op
from functools import reduce

from numba import njit

MAX_SEQUENCE_LENGTH = 4*6 #4 hours x 6 (time steps = 10 mins)


N_EVENT = 0 # number of event
for ns in range(1,MAX_SEQUENCE_LENGTH+1):
    n_ci = MAX_SEQUENCE_LENGTH-ns+1
    N_EVENT += n_ci


def build_NFA():
    nCr = np.diag(np.array(list(1 for _ in range(MAX_SEQUENCE_LENGTH+1)), dtype=np.float32))
    nCr[:,0] = 1
    for i in range(1, len(nCr)):
        for j in range(1, i):
            nCr[i,j] = nCr[i-1, j-1]+nCr[i-1, j]

    # compute exponents
    for ns in range(len(nCr)):
        for t in range(ns+1):
            nCr[ns,t] = nCr[ns,t]*(0.1**t)*(0.9**(ns-t))

    # build NFA
    NFA = np.zeros_like(nCr)
    for ns in range(NFA.shape[0]):
        for k in range(ns):
            NFA[ns, k] = 300*np.sum(nCr[ns, k:(ns+1)])
    return NFA

NFA = build_NFA()

# def nCr(n, r):
#     """Function calculates the number of combinations (n choose r)"""
#     r = min(r, n-r)
#     numer = reduce(op.mul, range(n, n-r, -1), 1)
#     denom = reduce(op.mul, range(1, r+1), 1)
#     return numer//denom


# def NFA(ns,k):
#     """Number of False Alarms"""
#     # B = 0
#     # for t in range(k,ns+1):
#     #     # B += nCr(ns,t)*(0.1**t)*(0.9**(ns-t))
#     #     B += nCr[ns,t]
#     # return 300*B
#     return 300*np.sum(nCr[ns, k:(ns+1)])
    

@njit
def contrario_detection(v_A_,epsilon=0.0091):
    """
    A contrario detection algorithms
    INPUT:
        v_A_: abnormal point indicator vector
        epsilon: threshold
    OUTPUT:
        v_anomalies: abnormal segment indicator vector

    """
    v_anomalies = np.zeros(len(v_A_))
    max_seq_len = min(MAX_SEQUENCE_LENGTH, len(v_A_))
    for d_ns in range(max_seq_len,0,-1):
        for d_ci in range(max_seq_len+1-d_ns):
            v_xi = v_A_[d_ci:d_ci+d_ns]
            d_k_xi = int(np.count_nonzero(v_xi))
            if NFA[d_ns,d_k_xi]<epsilon:
                v_anomalies[d_ci:d_ci+d_ns] = 1
    return v_anomalies



In [77]:

a = np.random.randint(0,2,34,dtype=np.uint8)
assert (contrario_detection_old(a) == contrario_detection(a)).all()
# for i in range(1000):
    

AssertionError: 

In [63]:
contrario_detection(np.random.randint(0,2,3333333333,dtype=np.uint8))

array([1., 1., 1., ..., 0., 0., 0.])

In [13]:
def f(x):
    return x.any()

def g(x):
    x = x.iloc[::-1].rolling(window=24, min_periods=0).apply(f).iloc[::-1]
    return x.any()


input.groupby(by='track_id')['outlier'].agg(g)

track_id
239       True
299       True
332       True
358      False
404       True
         ...  
65485     True
65493     True
65503     True
65509     True
65532     True
Name: outlier, Length: 1851, dtype: bool

In [ ]:
def apply(x):
    return 

for window in input.groupby(by='track_id')['outlier'].rolling(window=24):
    a = window

def f(x):
    return x.any()
a.iloc[::-1].rolling(window=24, min_periods=0).apply(f).iloc[::-1]